# BabyGrowth AI — ML Training Notebook

This notebook trains **RandomForest** models to predict future weight and height
for infants (0–36 months) using WHO growth reference data.

**Models produced:**
- `weight_model.pkl` — predicts weight 1 month ahead
- `height_model.pkl` — predicts height 1 month ahead

**Dataset:** `training_growth_dataset.csv` (generated from 500 synthetic children × 24 months)

In [ ]:
import sys
from pathlib import Path

# Adjust path to reach backend/ if running from project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'backend'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
%matplotlib inline

## 1. Load Dataset

In [ ]:
dataset_path = PROJECT_ROOT / 'backend' / 'training_growth_dataset.csv'
df = pd.read_csv(dataset_path)
print(f'Rows: {len(df):,}')
print(f'Columns: {list(df.columns)}\n')
df.head()

In [ ]:
print('Missing values per column:')
print(df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False))

In [ ]:
print(f'Unique children: {df.child_id.nunique()}')
print(f'Sex distribution:\n{df.sex.value_counts()}\n')
print(f'Age range: {df.age_months.min():.1f} – {df.age_months.max():.1f} months')

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

axes[0, 0].hist(df.current_weight.dropna(), bins=60, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Current Weight (kg)')

axes[0, 1].hist(df.current_height.dropna(), bins=60, color='coral', edgecolor='white')
axes[0, 1].set_title('Current Height (cm)')

axes[0, 2].hist(df.age_months.dropna(), bins=30, color='seagreen', edgecolor='white')
axes[0, 2].set_title('Age (months)')

axes[1, 0].hist(df.target_weight_next_month.dropna(), bins=60, color='steelblue', edgecolor='white', alpha=0.7)
axes[1, 0].set_title('Target Weight (next month, kg)')

axes[1, 1].hist(df.target_height_next_month.dropna(), bins=60, color='coral', edgecolor='white', alpha=0.7)
axes[1, 1].set_title('Target Height (next month, cm)')

axes[1, 2].hist(df.weight_percentile.dropna(), bins=40, color='purple', edgecolor='white')
axes[1, 2].set_title('Weight Percentile')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation with targets
corr_cols = ['age_months', 'current_weight', 'current_height',
             'weight_percentile', 'height_percentile', 'weight_zscore', 'height_zscore',
             'weight_gain_last_month', 'height_gain_last_month',
             'target_weight_next_month', 'target_height_next_month']

corr_df = df[corr_cols].dropna()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True)
plt.title('Feature Correlation Matrix')
plt.show()

## 3. Prepare Features

In [ ]:
FEATURES = [
    'age_months', 'birth_weight', 'birth_length',
    'current_weight', 'current_height', 'current_head_circumference',
    'weight_percentile', 'height_percentile', 'head_percentile',
    'weight_zscore', 'height_zscore', 'head_zscore',
    'weight_gain_last_month', 'height_gain_last_month',
]

TARGET_WEIGHT = 'target_weight_next_month'
TARGET_HEIGHT = 'target_height_next_month'

def prepare_data(df, target_col):
    data = df.dropna(subset=FEATURES + ['sex', target_col]).copy()
    data['sex_male'] = (data['sex'] == 'male').astype(int)
    X = data[FEATURES + ['sex_male']].values
    y = data[target_col].values
    return X, y, data

X_w, y_w, data_w = prepare_data(df, TARGET_WEIGHT)
X_h, y_h, data_h = prepare_data(df, TARGET_HEIGHT)

print(f'Weight samples: {len(X_w):,}')
print(f'Height samples: {len(X_h):,}')

## 4. Train Models

In [ ]:
def train_evaluate(X, y, name):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    r2 = r2_score(y_test, y_pred)

    print(f'\n=== {name} Model ===')
    print(f'  MAE : {mae:.4f}')
    print(f'  RMSE: {rmse:.4f}')
    print(f'  R²  : {r2:.4f}')

    return model, (X_train, X_test, y_train, y_test, y_pred, mae, rmse, r2)

weight_model, w_info = train_evaluate(X_w, y_w, 'Weight')
height_model, h_info = train_evaluate(X_h, y_h, 'Height')

## 5. Feature Importance

In [ ]:
all_features = FEATURES + ['sex_male']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model, name in zip(axes, [weight_model, height_model], ['Weight', 'Height']):
    imp = pd.DataFrame({'feature': all_features, 'importance': model.feature_importances_})
    imp = imp.sort_values('importance', ascending=True)
    ax.barh(imp['feature'], imp['importance'], color='steelblue' if name == 'Weight' else 'coral')
    ax.set_title(f'{name} Model — Feature Importance')
    ax.set_xlabel('Importance')

plt.tight_layout()
plt.show()

## 6. Prediction vs Actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (_, _, _, y_test, y_pred, mae, rmse, r2), name, color in zip(
    axes, [w_info, h_info], ['Weight (kg)', 'Height (cm)'], ['steelblue', 'coral']
):
    ax.scatter(y_test, y_pred, alpha=0.3, s=10, color=color)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', lw=1, label='Perfect prediction')
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(f'{name}\nMAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.3f}')
    ax.legend()
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
# Residual distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (_, _, _, y_test, y_pred, _, _, _), name, color in zip(
    axes, [w_info, h_info], ['Weight (kg)', 'Height (cm)'], ['steelblue', 'coral']
):
    residuals = y_test - y_pred
    ax.hist(residuals, bins=50, color=color, edgecolor='white', alpha=0.7)
    ax.axvline(0, color='red', linestyle='--', lw=1)
    ax.set_xlabel('Residual (Actual − Predicted)')
    ax.set_ylabel('Count')
    ax.set_title(f'{name} — Residuals  (std={residuals.std():.3f})')

plt.tight_layout()
plt.show()

## 7. Learning Curves

In [ ]:
def plot_learning_curve(model, X, y, name, color):
    train_sizes, train_scores, test_scores = learning_curve(
        model, X, y,
        train_sizes=np.linspace(0.1, 1.0, 8),
        cv=3, scoring='neg_mean_absolute_error', n_jobs=-1,
    )
    train_mae = -train_scores.mean(axis=1)
    test_mae = -test_scores.mean(axis=1)

    plt.figure(figsize=(8, 4))
    plt.plot(train_sizes, train_mae, 'o-', color=color, label='Train MAE')
    plt.plot(train_sizes, test_mae, 'o-', color='red', label='Test MAE')
    plt.xlabel('Training samples')
    plt.ylabel('MAE')
    plt.title(f'{name} — Learning Curve')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

plot_learning_curve(weight_model, X_w, y_w, 'Weight', 'steelblue')
plot_learning_curve(height_model, X_h, y_h, 'Height', 'coral')

## 8. Error by Age

In [ ]:
def error_by_age(model, X, y, data, name):
    y_pred = model.predict(X)
    error = np.abs(y - y_pred)
    ages = data['age_months'].values

    bins = np.arange(0, 25, 3)
    labels = [f'{int(b)}–{int(b+2)}mo' for b in bins[:-1]]
    idx = np.digitize(ages, bins) - 1

    plt.figure(figsize=(10, 4))
    means = [error[idx == i].mean() for i in range(len(bins) - 1)]
    plt.bar(labels, means, color='steelblue' if name == 'Weight' else 'coral', alpha=0.7)
    plt.xlabel('Age group')
    plt.ylabel('Mean Absolute Error')
    plt.title(f'{name} — MAE by Age Group')
    plt.grid(alpha=0.3, axis='y')
    plt.show()

error_by_age(weight_model, X_w, y_w, data_w, 'Weight')
error_by_age(height_model, X_h, y_h, data_h, 'Height')

## 9. Uncertainty (Confidence) Estimation

We use the **standard deviation across trees** as a proxy for prediction uncertainty.
The confidence score is: `1 − (std / (3 × RMSE))`, clamped to [0, 1].

In [ ]:
def tree_std(model, X):
    preds = np.array([tree.predict(X) for tree in model.estimators_])
    return preds.std(axis=0)

std_w = tree_std(weight_model, X_w[:1000])
std_h = tree_std(height_model, X_h[:1000])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(std_w, bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Weight — Tree Std Dev (first 1000 samples)')
axes[0].set_xlabel('Std dev (kg)')

axes[1].hist(std_h, bins=40, color='coral', edgecolor='white')
axes[1].set_title('Height — Tree Std Dev (first 1000 samples)')
axes[1].set_xlabel('Std dev (cm)')

plt.tight_layout()
plt.show()

print(f'Weight std:  mean={std_w.mean():.4f}  median={np.median(std_w):.4f}')
print(f'Height std:  mean={std_h.mean():.4f}  median={np.median(std_h):.4f}')

## 10. Save Models & Metrics

In [ ]:
models_dir = PROJECT_ROOT / 'backend' / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(weight_model, models_dir / 'weight_model.pkl')
joblib.dump(height_model, models_dir / 'height_model.pkl')
print(f'Models saved to {models_dir}')

import json
_, _, _, _, _, w_mae, w_rmse, w_r2 = w_info
_, _, _, _, _, h_mae, h_rmse, h_r2 = h_info

metrics = {
    'weight': {'mae': w_mae, 'rmse': w_rmse, 'r2': w_r2},
    'height': {'mae': h_mae, 'rmse': h_rmse, 'r2': h_r2},
}

with open(models_dir / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to models/metrics.json')

print(f'\nFinal summary:')
print(f'  Weight model — MAE={w_mae:.4f}  RMSE={w_rmse:.4f}  R²={w_r2:.4f}')
print(f'  Height model — MAE={h_mae:.4f}  RMSE={h_rmse:.4f}  R²={h_r2:.4f}')

---
**Notebook complete.** Both models are ready for inference via `services/ml/predict_growth.py`.